In [8]:
import json
import re
import uuid
import pandas as pd
from pathlib import Path

In [9]:
FILE_PATH = Path("/Users/promab/anaconda_projects/email_agent/data/processed/rag_files/CAR_T_products.xlsx")
SHEET_NAME = "Sheet1"
SOURCE_NAME = FILE_PATH.name
BUSINESS_LINE = "car_t"

raw_df = pd.read_excel(FILE_PATH, sheet_name="Sheet1")
raw_df.head()

,catalog_no,name,target_antigen,costimulatory_domain,construct,price_usd,total_time,unit,cell_number,marker,group_name,group_type,group_subtype,group_summary
0,PM-CAR1000,NaN,Mock,CD28,mock-TM28-CD28-CD3ζ,1259.0,NaN,NaN,NaN,NaN,CAR-T Cells,cell_product,NaN,NaN
1,PM-CAR1061,NaN,Mock,CD28,3xTF-TM28-CD28-CD3ζ,1259.0,NaN,NaN,NaN,NaN,CAR-T Cells,cell_product,NaN,NaN
2,PM-CAR1086,NaN,Mock,4-1BB,3xTF-TM-4-1BB-CD3ζ,1259.0,NaN,NaN,NaN,NaN,CAR-T Cells,cell_product,NaN,NaN
3,PM-CAR1079,NaN,alpha Fetaprotein,4-1BB,αFPscFv-TM-4-1BB-CD3ζ,1259.0,NaN,NaN,NaN,NaN,CAR-T Cells,cell_product,NaN,NaN
4,PM-CAR1058,NaN,B7H4,CD28,B7H4scFv-TM28-CD28-CD3ζ,1259.0,NaN,NaN,NaN,NaN,CAR-T Cells,cell_product,NaN,NaN


In [10]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   catalog_no            143 non-null    object 
 1   name                  49 non-null     str    
 2   target_antigen        93 non-null     str    
 3   costimulatory_domain  94 non-null     str    
 4   construct             94 non-null     str    
 5   price_usd             143 non-null    float64
 6   total_time            5 non-null      object 
 7   unit                  9 non-null      str    
 8   cell_number           35 non-null     str    
 9   marker                23 non-null     str    
 10  group_name            138 non-null    str    
 11  group_type            138 non-null    str    
 12  group_subtype         40 non-null     str    
 13  group_summary         44 non-null     str    
dtypes: float64(1), object(2), str(11)
memory usage: 32.0+ KB


In [11]:
def norm(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text or None


def normalize_title(text):
    normalized = norm(text)
    if not normalized:
        return None
    normalized = normalized.lower()
    normalized = re.sub(r"[^a-z0-9]+", " ", normalized)
    return re.sub(r"\s+", " ", normalized).strip()


def extract_lead_time_days(value):
    text = norm(value)
    if not text:
        return None
    match = re.search(r"(\d+)", text)
    return int(match.group(1)) if match else None


def build_title(row):
    explicit_name = norm(row.get("name"))
    if explicit_name:
        return explicit_name

    parts = [
        norm(row.get("group_name")),
        norm(row.get("target_antigen")),
        norm(row.get("costimulatory_domain")),
        norm(row.get("catalog_no")),
    ]
    return " ".join(part for part in parts if part)


def build_description(row):
    summary = norm(row.get("group_summary"))
    if summary:
        return summary

    parts = [
        f"Target antigen: {norm(row.get('target_antigen'))}" if norm(row.get("target_antigen")) else None,
        f"Costimulatory domain: {norm(row.get('costimulatory_domain'))}" if norm(row.get("costimulatory_domain")) else None,
        f"Construct: {norm(row.get('construct'))}" if norm(row.get("construct")) else None,
        f"Cell number: {norm(row.get('cell_number'))}" if norm(row.get("cell_number")) else None,
        f"Marker: {norm(row.get('marker'))}" if norm(row.get("marker")) else None,
        f"Total time: {norm(row.get('total_time'))}" if norm(row.get("total_time")) else None,
    ]
    return " | ".join(part for part in parts if part) or None


def build_keywords(row):
    values = [
        norm(row.get("catalog_no")),
        norm(row.get("name")),
        norm(row.get("target_antigen")),
        norm(row.get("costimulatory_domain")),
        norm(row.get("construct")),
        norm(row.get("marker")),
        norm(row.get("group_name")),
        norm(row.get("group_type")),
        norm(row.get("group_subtype")),
        norm(row.get("unit")),
        norm(row.get("cell_number")),
    ]
    deduped = []
    seen = set()
    for value in values:
        if not value:
            continue
        lowered = value.lower()
        if lowered in seen:
            continue
        seen.add(lowered)
        deduped.append(value)
    return deduped


def build_search_text(title, description, keywords, row):
    parts = [
        norm(row.get("catalog_no")),
        title,
        description,
        " ".join(keywords) if keywords else None,
        norm(row.get("group_name")),
        norm(row.get("group_type")),
        norm(row.get("group_subtype")),
        norm(row.get("target_antigen")),
        norm(row.get("construct")),
    ]
    return " | ".join(part for part in parts if part)


raw_df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

records = []
for idx, row in raw_df.iterrows():
    title = build_title(row)
    description = build_description(row)
    keywords = build_keywords(row)

    records.append(
        {
            "id": str(uuid.uuid4()),
            "source_id": None,
            "source_type": "excel",
            "source_name": SOURCE_NAME,
            "source_sheet": SHEET_NAME,
            "source_row_number": idx + 2,
            "business_line": BUSINESS_LINE,
            "record_type": norm(row.get("group_type")) or "product",
            "product_family": norm(row.get("group_name")),
            "catalog_no": norm(row.get("catalog_no")),
            "title": title,
            "normalized_title": normalize_title(title),
            "description": description,
            "aliases": [],
            "keywords": keywords,
            "applications": [],
            "species_reactivity": [],
            "target": norm(row.get("target_antigen")),
            "gene_id": None,
            "gene_accession": None,
            "swissprot": None,
            "antibody_type": None,
            "clone_name": None,
            "isotype_or_class": None,
            "format_size": norm(row.get("unit")),
            "unit_price": float(row["price_usd"]) if pd.notna(row.get("price_usd")) else None,
            "currency": "USD",
            "availability_status": "catalog_available",
            "lead_time_days": extract_lead_time_days(row.get("total_time")),
            "name": norm(row.get("name")),
            "target_antigen": norm(row.get("target_antigen")),
            "costimulatory_domain": norm(row.get("costimulatory_domain")),
            "construct": norm(row.get("construct")),
            "total_time": norm(row.get("total_time")),
            "unit": norm(row.get("unit")),
            "cell_number": norm(row.get("cell_number")),
            "marker": norm(row.get("marker")),
            "group_name": norm(row.get("group_name")),
            "group_type": norm(row.get("group_type")),
            "group_subtype": norm(row.get("group_subtype")),
            "group_summary": norm(row.get("group_summary")),
            "raw_metadata": {
                column: (None if pd.isna(value) else value)
                for column, value in row.to_dict().items()
            },
            "search_text": build_search_text(title, description, keywords, row),
            "embedding": None,
            "is_active": True,
        }
    )

catalog_df = pd.DataFrame(records)

preview_columns = [
    "catalog_no",
    "name",
    "title",
    "record_type",
    "product_family",
    "target_antigen",
    "costimulatory_domain",
    "construct",
    "unit_price",
    "currency",
    "lead_time_days",
    "keywords",
    "description",
    "search_text",
]

compare_df = pd.DataFrame(
    {
        "catalog_no_raw": raw_df["catalog_no"],
        "name_raw": raw_df["name"],
        "group_name_raw": raw_df["group_name"],
        "group_type_raw": raw_df["group_type"],
        "target_antigen_raw": raw_df["target_antigen"],
        "costimulatory_domain_raw": raw_df["costimulatory_domain"],
        "construct_raw": raw_df["construct"],
        "price_usd_raw": raw_df["price_usd"],
        "title_new": catalog_df["title"],
        "record_type_new": catalog_df["record_type"],
        "product_family_new": catalog_df["product_family"],
        "unit_price_new": catalog_df["unit_price"],
        "keywords_new": catalog_df["keywords"],
    }
)

print("raw_df shape:", raw_df.shape)
print("catalog_df shape:", catalog_df.shape)

raw_df.head()
catalog_df[preview_columns].head(10)
compare_df.head(10)

raw_df shape: (143, 14)
catalog_df shape: (143, 45)


,catalog_no_raw,name_raw,group_name_raw,group_type_raw,target_antigen_raw,costimulatory_domain_raw,construct_raw,price_usd_raw,title_new,record_type_new,product_family_new,unit_price_new,keywords_new
0,PM-CAR1000,NaN,CAR-T Cells,cell_product,Mock,CD28,mock-TM28-CD28-CD3ζ,1259.0,CAR-T Cells Mock CD28 PM-CAR1000,cell_product,CAR-T Cells,1259.0,"[PM-CAR1000, Mock, CD28, mock-TM28-CD28-CD3ζ, ..."
1,PM-CAR1061,NaN,CAR-T Cells,cell_product,Mock,CD28,3xTF-TM28-CD28-CD3ζ,1259.0,CAR-T Cells Mock CD28 PM-CAR1061,cell_product,CAR-T Cells,1259.0,"[PM-CAR1061, Mock, CD28, 3xTF-TM28-CD28-CD3ζ, ..."
2,PM-CAR1086,NaN,CAR-T Cells,cell_product,Mock,4-1BB,3xTF-TM-4-1BB-CD3ζ,1259.0,CAR-T Cells Mock 4-1BB PM-CAR1086,cell_product,CAR-T Cells,1259.0,"[PM-CAR1086, Mock, 4-1BB, 3xTF-TM-4-1BB-CD3ζ, ..."
3,PM-CAR1079,NaN,CAR-T Cells,cell_product,alpha Fetaprotein,4-1BB,αFPscFv-TM-4-1BB-CD3ζ,1259.0,CAR-T Cells alpha Fetaprotein 4-1BB PM-CAR1079,cell_product,CAR-T Cells,1259.0,"[PM-CAR1079, alpha Fetaprotein, 4-1BB, αFPscFv..."
4,PM-CAR1058,NaN,CAR-T Cells,cell_product,B7H4,CD28,B7H4scFv-TM28-CD28-CD3ζ,1259.0,CAR-T Cells B7H4 CD28 PM-CAR1058,cell_product,CAR-T Cells,1259.0,"[PM-CAR1058, B7H4, CD28, B7H4scFv-TM28-CD28-CD..."
5,PM-CAR1031,NaN,CAR-T Cells,cell_product,BCMA,CD28,BCMAscFv-TM28-CD28-CD3ζ,1259.0,CAR-T Cells BCMA CD28 PM-CAR1031,cell_product,CAR-T Cells,1259.0,"[PM-CAR1031, BCMA, CD28, BCMAscFv-TM28-CD28-CD..."
6,PM-CAR1033,NaN,CAR-T Cells,cell_product,BCMA,CD28,huBCMAscFv-TM28-CD28-CD3ζ,1259.0,CAR-T Cells BCMA CD28 PM-CAR1033,cell_product,CAR-T Cells,1259.0,"[PM-CAR1033, BCMA, CD28, huBCMAscFv-TM28-CD28-..."
7,PM-CAR1037,NaN,CAR-T Cells,cell_product,BCMA,4-1BB,BCMAscFv-TM8-4-1BB-CD3ζ,1259.0,CAR-T Cells BCMA 4-1BB PM-CAR1037,cell_product,CAR-T Cells,1259.0,"[PM-CAR1037, BCMA, 4-1BB, BCMAscFv-TM8-4-1BB-C..."
8,PM-CAR1051,NaN,CAR-T Cells,cell_product,BCMA,CD28,BCMAscFv-TM-CD28-CD3ζ,1259.0,CAR-T Cells BCMA CD28 PM-CAR1051,cell_product,CAR-T Cells,1259.0,"[PM-CAR1051, BCMA, CD28, BCMAscFv-TM-CD28-CD3ζ..."
9,PM-CAR1093,NaN,CAR-T Cells,cell_product,CA-125,4-1BB,CA-125 scFv-4-1BB-CD3ζ,1259.0,CAR-T Cells CA-125 4-1BB PM-CAR1093,cell_product,CAR-T Cells,1259.0,"[PM-CAR1093, CA-125, 4-1BB, CA-125 scFv-4-1BB-..."


In [12]:
catalog_df[preview_columns].head(10)

,catalog_no,name,title,record_type,product_family,target_antigen,costimulatory_domain,construct,unit_price,currency,lead_time_days,keywords,description,search_text
0,PM-CAR1000,NaN,CAR-T Cells Mock CD28 PM-CAR1000,cell_product,CAR-T Cells,Mock,CD28,mock-TM28-CD28-CD3ζ,1259.0,USD,NaN,"[PM-CAR1000, Mock, CD28, mock-TM28-CD28-CD3ζ, ...",Target antigen: Mock | Costimulatory domain: C...,PM-CAR1000 | CAR-T Cells Mock CD28 PM-CAR1000 ...
1,PM-CAR1061,NaN,CAR-T Cells Mock CD28 PM-CAR1061,cell_product,CAR-T Cells,Mock,CD28,3xTF-TM28-CD28-CD3ζ,1259.0,USD,NaN,"[PM-CAR1061, Mock, CD28, 3xTF-TM28-CD28-CD3ζ, ...",Target antigen: Mock | Costimulatory domain: C...,PM-CAR1061 | CAR-T Cells Mock CD28 PM-CAR1061 ...
2,PM-CAR1086,NaN,CAR-T Cells Mock 4-1BB PM-CAR1086,cell_product,CAR-T Cells,Mock,4-1BB,3xTF-TM-4-1BB-CD3ζ,1259.0,USD,NaN,"[PM-CAR1086, Mock, 4-1BB, 3xTF-TM-4-1BB-CD3ζ, ...",Target antigen: Mock | Costimulatory domain: 4...,PM-CAR1086 | CAR-T Cells Mock 4-1BB PM-CAR1086...
3,PM-CAR1079,NaN,CAR-T Cells alpha Fetaprotein 4-1BB PM-CAR1079,cell_product,CAR-T Cells,alpha Fetaprotein,4-1BB,αFPscFv-TM-4-1BB-CD3ζ,1259.0,USD,NaN,"[PM-CAR1079, alpha Fetaprotein, 4-1BB, αFPscFv...",Target antigen: alpha Fetaprotein | Costimulat...,PM-CAR1079 | CAR-T Cells alpha Fetaprotein 4-1...
4,PM-CAR1058,NaN,CAR-T Cells B7H4 CD28 PM-CAR1058,cell_product,CAR-T Cells,B7H4,CD28,B7H4scFv-TM28-CD28-CD3ζ,1259.0,USD,NaN,"[PM-CAR1058, B7H4, CD28, B7H4scFv-TM28-CD28-CD...",Target antigen: B7H4 | Costimulatory domain: C...,PM-CAR1058 | CAR-T Cells B7H4 CD28 PM-CAR1058 ...
5,PM-CAR1031,NaN,CAR-T Cells BCMA CD28 PM-CAR1031,cell_product,CAR-T Cells,BCMA,CD28,BCMAscFv-TM28-CD28-CD3ζ,1259.0,USD,NaN,"[PM-CAR1031, BCMA, CD28, BCMAscFv-TM28-CD28-CD...",Target antigen: BCMA | Costimulatory domain: C...,PM-CAR1031 | CAR-T Cells BCMA CD28 PM-CAR1031 ...
6,PM-CAR1033,NaN,CAR-T Cells BCMA CD28 PM-CAR1033,cell_product,CAR-T Cells,BCMA,CD28,huBCMAscFv-TM28-CD28-CD3ζ,1259.0,USD,NaN,"[PM-CAR1033, BCMA, CD28, huBCMAscFv-TM28-CD28-...",Target antigen: BCMA | Costimulatory domain: C...,PM-CAR1033 | CAR-T Cells BCMA CD28 PM-CAR1033 ...
7,PM-CAR1037,NaN,CAR-T Cells BCMA 4-1BB PM-CAR1037,cell_product,CAR-T Cells,BCMA,4-1BB,BCMAscFv-TM8-4-1BB-CD3ζ,1259.0,USD,NaN,"[PM-CAR1037, BCMA, 4-1BB, BCMAscFv-TM8-4-1BB-C...",Target antigen: BCMA | Costimulatory domain: 4...,PM-CAR1037 | CAR-T Cells BCMA 4-1BB PM-CAR1037...
8,PM-CAR1051,NaN,CAR-T Cells BCMA CD28 PM-CAR1051,cell_product,CAR-T Cells,BCMA,CD28,BCMAscFv-TM-CD28-CD3ζ,1259.0,USD,NaN,"[PM-CAR1051, BCMA, CD28, BCMAscFv-TM-CD28-CD3ζ...",Target antigen: BCMA | Costimulatory domain: C...,PM-CAR1051 | CAR-T Cells BCMA CD28 PM-CAR1051 ...
9,PM-CAR1093,NaN,CAR-T Cells CA-125 4-1BB PM-CAR1093,cell_product,CAR-T Cells,CA-125,4-1BB,CA-125 scFv-4-1BB-CD3ζ,1259.0,USD,NaN,"[PM-CAR1093, CA-125, 4-1BB, CA-125 scFv-4-1BB-...",Target antigen: CA-125 | Costimulatory domain:...,PM-CAR1093 | CAR-T Cells CA-125 4-1BB PM-CAR10...


In [ ]:
pkill -f "uvicorn"
uvicorn src.main:app --reload